### Time Series forcasting

https://www.kaggle.com/competitions/store-sales-time-series-forecasting

In [1]:
!pip install -U statsmodels
!curl -L "https://drive.google.com/u/1/uc?id=1DgD3Ue7P1m5nb0ssn2f2nMsfkYtZVuK3&export=download" --output dataset.zip
! yes | unzip dataset.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 21.3M  100 21.3M    0     0  4223k      0  0:00:05  0:00:05 --:--:-- 5788k
Archive:  dataset.zip
  inflating: holidays_events.csv     
  inflating: oil.csv                 
  inflating: sample_submission.csv   
  inflating: stores.csv              
  inflating: test.csv                
  inflating: train.csv               
  inflating: transactions.csv        


In [1]:
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (15,3)
plt.rcParams['figure.dpi'] = 200
import numpy as np

In [2]:
# Load datasets
import pandas as pd

train_df = pd.read_csv("train.csv", parse_dates=['date'], infer_datetime_format=True)
test_df = pd.read_csv("test.csv", parse_dates=['date'], infer_datetime_format=True)
stores_df = pd.read_csv("stores.csv")
oil_df = pd.read_csv("oil.csv", parse_dates=['date'], infer_datetime_format=True)
holiday_df = pd.read_csv("holidays_events.csv", parse_dates=['date'], infer_datetime_format=True)
transactions_df = pd.read_csv("transactions.csv")
sample_submission_df = pd.read_csv("sample_submission.csv")

<ipython-input-2-75ceae71fa8a>:4: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  train_df = pd.read_csv("train.csv", parse_dates=['date'], infer_datetime_format=True)
<ipython-input-2-75ceae71fa8a>:5: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  test_df = pd.read_csv("test.csv", parse_dates=['date'], infer_datetime_format=True)
<ipython-input-2-75ceae71fa8a>:7: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-con

# 1. Simple regression

To treat the who process as a simple regression with no knowledge about time.

**Task**

1. Split the train_df in X_train, X_test, y_train, y_test
2. Remember to split by time, with no shuffling
3. Do all feature engineering required
4. Run the regression with xgboost
5. Perform model evaluation with RMSLE

In [3]:
train_df

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.000,0
1,1,2013-01-01,1,BABY CARE,0.000,0
2,2,2013-01-01,1,BEAUTY,0.000,0
3,3,2013-01-01,1,BEVERAGES,0.000,0
4,4,2013-01-01,1,BOOKS,0.000,0
...,...,...,...,...,...,...
3000883,3000883,2017-08-15,9,POULTRY,438.133,0
3000884,3000884,2017-08-15,9,PREPARED FOODS,154.553,1
3000885,3000885,2017-08-15,9,PRODUCE,2419.729,148
3000886,3000886,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000,8


In [23]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [39]:
from sklearn.model_selection import train_test_split

X = train_df.drop(['id', 'sales'], axis=1)
y = train_df['sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [40]:
X_train['dow'] = X_train.date.dt.day_of_week

In [41]:
X_train

,date,store_nbr,family,onpromotion,dow
2886109,2017-06-12,38,POULTRY,0,0
1308436,2015-01-07,21,LADIESWEAR,0,2
2505116,2016-11-09,48,LAWN AND GARDEN,0,2
916601,2014-05-31,27,PET SUPPLIES,0,5
470668,2013-09-22,15,"LIQUOR,WINE,BEER",0,6
...,...,...,...,...,...
1692743,2015-08-10,54,DAIRY,0,0
2356330,2016-08-18,23,SCHOOL AND OFFICE SUPPLIES,0,3
2229084,2016-06-07,53,AUTOMOTIVE,0,1
2768307,2017-04-07,33,BEVERAGES,15,4


In [42]:
encoder = OrdinalEncoder()

X_train_onehot = encoder.fit_transform(X_train[['store_nbr', 'family']])

In [43]:
X_train[encoder.get_feature_names_out()] = X_train_onehot

In [46]:
encoder.get_feature_names_out()

array(['store_nbr', 'family'], dtype=object)

In [48]:
X_train.drop(['date'], inplace=True, axis=1)

In [50]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()

model.fit(X_train, y_train)

RandomForestRegressor()

In [51]:
X_test['dow'] = X_test.date.dt.day_of_week
X_test_onehot = encoder.transform(X_test[['store_nbr', 'family']])
X_test[encoder.get_feature_names_out()] = X_test_onehot
X_test.drop(['date'], inplace=True, axis=1)

In [52]:
y_pred = model.predict(X_test)

In [53]:
model.score(X_test, y_test)

0.871441066457026

In [54]:
y_pred

array([2.61957000e+03, 0.00000000e+00, 3.68593297e+02, ...,
       2.26837811e+02, 5.69392941e+00, 1.42442476e-01])

In [55]:
y_pred >= 0

array([ True,  True,  True, ...,  True,  True,  True])

In [56]:
y_pred * (y_pred > 0)

array([2.61957000e+03, 0.00000000e+00, 3.68593297e+02, ...,
       2.26837811e+02, 5.69392941e+00, 1.42442476e-01])

In [57]:
from sklearn.metrics import mean_squared_log_error

mean_squared_log_error(y_pred * (y_pred > 0), y_test) ** 0.5

1.2817919311754504

## 2. Add holidays, oil price and other information

Let's add other information to X.

**Task**

1. Prepare X with holiday information
2. Prepare X with oil price

*Thinking:* Are these information useful?

## 3. Add Seasonality

On top of the features used in the previous section, can we also add the fourier features?

**Task**
1. Create yearly fourier features of order 10
2. Add the features to the previous X_train and X_test
3. Rerun the xgboost model
4. Check the change in RMSLE

*Thinking:* Can we recompose the waves from the fitted model?